# DeepOMAPNet — Training Notebook

End-to-end workflow: generate synthetic CITE-seq data → build graphs → train → evaluate.

In [ ]:
import sys, os, importlib, warnings
warnings.filterwarnings('ignore')

current_dir = os.getcwd()
project_root = os.path.dirname(current_dir)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import torch
import numpy as np
import matplotlib.pyplot as plt

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')

## 1. Load Data (Synthetic CITE-seq)

In [ ]:
from scripts.data_provider.synthetic_citeseq import generate_citeseq_dataset, CELL_TYPE_NAMES

# Generate synthetic CITE-seq dataset (replace with real data loader if available)
dataset = generate_citeseq_dataset(n_normal=1500, n_aml=1500, seed=42)

print(f'Total cells  : {dataset.n_cells}')
print(f'RNA features : {dataset.n_genes}')
print(f'ADT proteins : {dataset.n_adts}')
print(f'AML cells    : {dataset.aml_label.sum()} ({dataset.aml_label.mean()*100:.0f}%)')
print()
print('Cell-type breakdown:')
for i, name in enumerate(CELL_TYPE_NAMES):
    count = (dataset.celltype_label == i).sum()
    print(f'  {name:<12}: {count:4d} cells ({count/dataset.n_cells*100:.1f}%)')

# Load adata_gene_train.h5ad from /projects/vanaja_lab/satya/Datasets

In [ ]:
import anndata as ad
import numpy as np

base_path = "/projects/vanaja_lab/satya/Datasets"

# ── 1. Load each modality separately ─────────────────────────────────────────
# RNA and ADT have different feature spaces (genes vs proteins) and must NEVER
# be concatenated together.

rna_control = ad.read_h5ad(f"{base_path}/GSMControlRNA.h5ad")
rna_amla    = ad.read_h5ad(f"{base_path}/AMLARNA.h5ad")
rna_amlb    = ad.read_h5ad(f"{base_path}/AMLBRNA.h5ad")

adt_control = ad.read_h5ad(f"{base_path}/ControlADT.h5ad")
adt_amla    = ad.read_h5ad(f"{base_path}/AMLAADT.h5ad")
adt_amlb    = ad.read_h5ad(f"{base_path}/AMLBADT.h5ad")

for obj, tag in [(rna_control, 'Control'), (rna_amla, 'AMLA'), (rna_amlb, 'AMLB')]:
    obj.obs['source'] = tag
for obj, tag in [(adt_control, 'Control'), (adt_amla, 'AMLA'), (adt_amlb, 'AMLB')]:
    obj.obs['source'] = tag

print("RNA files:")
for name, obj in [("Control", rna_control), ("AMLA", rna_amla), ("AMLB", rna_amlb)]:
    print(f"  {name}: {obj.shape}")
print("ADT files:")
for name, obj in [("Control", adt_control), ("AMLA", adt_amla), ("AMLB", adt_amlb)]:
    print(f"  {name}: {obj.shape}")

# ── 2. Concatenate within each modality ──────────────────────────────────────
# join='inner' keeps only features (genes / proteins) present in all batches.
rna_combined = ad.concat(
    [rna_control, rna_amla, rna_amlb],
    join='inner', label='source', keys=['Control', 'AMLA', 'AMLB']
)
adt_combined = ad.concat(
    [adt_control, adt_amla, adt_amlb],
    join='inner', label='source', keys=['Control', 'AMLA', 'AMLB']
)
rna_combined.obs_names_make_unique()
adt_combined.obs_names_make_unique()

assert rna_combined.n_obs == adt_combined.n_obs, (
    f"Cell count mismatch: RNA={rna_combined.n_obs}, ADT={adt_combined.n_obs}. "
    "Each RNA file must be paired with its ADT file (same cells, same order)."
)

print(f"\nRNA combined : {rna_combined.shape}  ({rna_combined.n_vars} genes)")
print(f"ADT combined : {adt_combined.shape}  ({adt_combined.n_vars} proteins)")

# ── 3. Build AML labels (0 = Normal/Control, 1 = AML) ────────────────────────
aml_labels = (rna_combined.obs['source'] != 'Control').astype(int).values
print(f"\nAML labels  — Control: {(aml_labels == 0).sum()}  |  AML: {(aml_labels == 1).sum()}")

# ── 4. Cell-type labels (optional — set to None if column is absent) ──────────
_ct_col = 'Cell_type_identity'
if _ct_col in rna_combined.obs.columns:
    from sklearn.preprocessing import LabelEncoder
    cell_labels = LabelEncoder().fit_transform(rna_combined.obs[_ct_col].values)
    num_cell_types = int(cell_labels.max()) + 1
    print(f"Cell types  : {num_cell_types} unique labels")
else:
    cell_labels    = None
    num_cell_types = None
    print("Cell-type labels not found in obs — cell-type classification disabled.")

## 2. Build PyTorch Geometric Graphs

The trainer handles the 80/10/10 train/val/test split internally via node masks.
We pass the full combined RNA and ADT objects — no manual splitting needed.

In [ ]:
from scripts.data_provider.graph_data_builder import build_pyg_data

print("Building RNA graph...")
rna_pyg = build_pyg_data(rna_combined)
print(f"  {rna_pyg}")

print("Building ADT graph...")
adt_pyg = build_pyg_data(adt_combined)
print(f"  {adt_pyg}")

assert rna_pyg.num_nodes == adt_pyg.num_nodes, "Node count mismatch!"
print(f"\nGraphs built — {rna_pyg.num_nodes} cells, "
      f"{rna_pyg.num_edges} RNA edges, {adt_pyg.num_edges} ADT edges")

## 4. Train the Model

In [ ]:
from scripts.trainer.gat_trainer import train_gat_transformer_fusion



training_config = dict(
    epochs=100,
    seed=42,
    learning_rate=1e-3,
    weight_decay=1e-4,
    dropout_rate=0.4,
    hidden_channels=64,
    num_heads=4,
    num_attention_heads=4,
    num_layers=2,
    use_mixed_precision=torch.cuda.is_available(),
    early_stopping_patience=10,
)

(
    trained_model,
    rna_data_with_masks,
    adt_data_with_masks,
    training_history,
    adt_mean, adt_std,
    node_degrees_rna, node_degrees_adt,
    clustering_coeffs_rna, clustering_coeffs_adt,
) = train_gat_transformer_fusion(
    rna_data=rna_pyg,
    adt_data=adt_pyg,
    rna_anndata=train_rna,
    adt_anndata=train_adt,
    celltype_weight=0.5,
    **training_config,
)

print(f"\nBest val  R²: {max(training_history['val_R2']):.4f}")
print(f"Final test R²: {training_history['test_R2'][-1]:.4f}")

from scripts.trainer.gat_trainer import train_gat_transformer_fusion

training_config = dict(
    epochs=100,
    seed=42,
    learning_rate=1e-3,
    weight_decay=1e-4,
    dropout_rate=0.4,
    hidden_channels=64,
    num_heads=4,
    num_attention_heads=4,
    num_layers=2,
    use_mixed_precision=torch.cuda.is_available(),
    early_stopping_patience=10,
)

(
    trained_model,
    rna_data_with_masks,
    adt_data_with_masks,
    training_history,
    adt_mean, adt_std,
    node_degrees_rna, node_degrees_adt,
    clustering_coeffs_rna, clustering_coeffs_adt,
) = train_gat_transformer_fusion(
    rna_data=rna_pyg,
    adt_data=adt_pyg,
    aml_labels=aml_labels,
    celltype_labels=cell_labels,        # None if not available
    num_cell_types=num_cell_types,      # None if not available
    celltype_weight=0.5,
    stratify_labels=aml_labels,         # keeps AML ratio balanced across splits
    **training_config,
)

print(f"\nBest val  R²: {max(training_history['val_R2']):.4f}")
print(f"Final test R²: {training_history['test_R2'][-1]:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(training_history['train_loss'])
axes[0].set_title('Training Loss'); axes[0].set_xlabel('Epoch')

axes[1].plot(training_history['val_MSE'],  label='Val')
axes[1].plot(training_history['test_MSE'], label='Test')
axes[1].set_title('ADT MSE'); axes[1].set_xlabel('Epoch'); axes[1].legend()

axes[2].plot(training_history['val_R2'],  label='Val')
axes[2].plot(training_history['test_R2'], label='Test')
axes[2].set_title('ADT R²'); axes[2].set_xlabel('Epoch'); axes[2].legend()

plt.tight_layout()
plt.show()

## 6. Evaluate on Held-Out Test Set

In [ ]:
import scipy.sparse as sp
from sklearn.metrics import r2_score, mean_squared_error, accuracy_score, roc_auc_score

device = next(trained_model.parameters()).device

# Build test graph
print('Building test graph...')
rna_test_pyg = build_pyg_data(rna_test)

# Prepare features
X_test = rna_test.X.toarray() if sp.issparse(rna_test.X) else np.array(rna_test.X)
rna_features  = torch.tensor(X_test, dtype=torch.float32).to(device)
edge_index_rna = rna_test_pyg.edge_index.to(device)

# Inference
trained_model.eval()
with torch.no_grad():
    adt_pred, aml_pred, fused_emb = trained_model(
        x=rna_features,
        edge_index_rna=edge_index_rna,
        edge_index_adt=None,
    )

adt_pred_np = adt_pred.cpu().numpy()
adt_true_np = adt_test.X if not sp.issparse(adt_test.X) else adt_test.X.toarray()

# ADT regression metrics
r2  = r2_score(adt_true_np, adt_pred_np)
mse = mean_squared_error(adt_true_np, adt_pred_np)

# AML classification metrics
aml_probs  = torch.sigmoid(aml_pred).cpu().numpy().flatten()
aml_binary = (aml_probs > 0.5).astype(int)
acc = accuracy_score(aml_labels_test, aml_binary)
auc = roc_auc_score(aml_labels_test, aml_probs)

print('=== Held-out Test Results ===')
print(f'ADT  R²      : {r2:.4f}')
print(f'ADT  MSE     : {mse:.4f}')
print(f'AML  Accuracy: {acc:.4f}')
print(f'AML  AUC-ROC : {auc:.4f}')

from sklearn.metrics import r2_score, mean_squared_error, accuracy_score, roc_auc_score

device = next(trained_model.parameters()).device

# Move everything the model needs to the same device
rna_inf = rna_data_with_masks.to(device)
adt_inf = adt_data_with_masks.to(device)
test_mask    = rna_inf.test_mask
adt_mean_dev = adt_mean.to(device)
adt_std_dev  = adt_std.to(device)
nd_rna = node_degrees_rna.to(device)
nd_adt = node_degrees_adt.to(device)
cc_rna = clustering_coeffs_rna.to(device)
cc_adt = clustering_coeffs_adt.to(device)

trained_model.eval()
with torch.no_grad():
    adt_pred, aml_pred, fused_emb = trained_model(
        x=rna_inf.x,
        edge_index_rna=rna_inf.edge_index,
        edge_index_adt=adt_inf.edge_index,
        node_degrees_rna=nd_rna,
        node_degrees_adt=nd_adt,
        clustering_coeffs_rna=cc_rna,
        clustering_coeffs_adt=cc_adt,
    )

# Denormalise ADT predictions
adt_pred_np = (adt_pred[test_mask] * adt_std_dev + adt_mean_dev).cpu().numpy()
adt_true_np = (adt_inf.x[test_mask]  * adt_std_dev + adt_mean_dev).cpu().numpy()

# ADT regression metrics
r2  = r2_score(adt_true_np.ravel(), adt_pred_np.ravel())
mse = mean_squared_error(adt_true_np, adt_pred_np)

# AML classification metrics
aml_probs  = torch.sigmoid(aml_pred[test_mask]).cpu().numpy().flatten()
aml_binary = (aml_probs > 0.5).astype(int)
aml_true   = torch.tensor(aml_labels, dtype=torch.float32)[test_mask.cpu()].numpy().astype(int)
acc = accuracy_score(aml_true, aml_binary)
auc = roc_auc_score(aml_true, aml_probs)

print("=== Held-out Test Results ===")
print(f"ADT  R²      : {r2:.4f}")
print(f"ADT  MSE     : {mse:.4f}")
print(f"AML  Accuracy: {acc:.4f}")
print(f"AML  AUC-ROC : {auc:.4f}")

In [ ]:
from scipy.stats import pearsonr

n_proteins = adt_pred_np.shape[1]
protein_names = dataset.adt_names
correlations = [pearsonr(adt_true_np[:, p], adt_pred_np[:, p])[0] for p in range(n_proteins)]

# Bar chart
fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(protein_names, correlations)
ax.axhline(0, color='k', linewidth=0.5)
ax.set_ylabel('Pearson r')
ax.set_title('Per-Protein Prediction Correlation (Test Set)')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.show()

print(f'Mean correlation: {np.mean(correlations):.4f}')
print(f'Best protein   : {protein_names[int(np.argmax(correlations))]} ({max(correlations):.4f})')

from scipy.stats import pearsonr

protein_names = list(adt_combined.var_names)
n_proteins    = adt_pred_np.shape[1]

correlations = [
    pearsonr(adt_true_np[:, p], adt_pred_np[:, p])[0]
    for p in range(n_proteins)
]

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(protein_names, correlations)
ax.axhline(0, color='k', linewidth=0.5)
ax.set_ylabel("Pearson r")
ax.set_title("Per-Protein Prediction Correlation (Test Set)")
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.show()

best_idx = int(np.argmax(correlations))
print(f"Mean correlation : {np.mean(correlations):.4f}")
print(f"Best protein     : {protein_names[best_idx]} ({correlations[best_idx]:.4f}"
      f"  worst: {protein_names[int(np.argmin(correlations))]} ({min(correlations):.4f})")

In [ ]:
torch.save(trained_model.state_dict(), 'DeepOMAPNet_weights.pth')
print('Model saved to DeepOMAPNet_weights.pth')